# Stage 3: the basis

The `LinearOperator` from [`03_linearization`](../03_linearization.ipynb) carries no basis, and
without one it has no matrix representation. A basis supplies what is missing: which functions of
the state are the coordinates, how many there are, and what happens to anything above the
truncation degree.

`lin_ode` provides three, at truncation degree $K$.

- **`CarlemanBasis`**: monomials in the state,
  $y_{\boldsymbol\alpha}(u)=\prod_i u_i^{\alpha_i}$ with $|\boldsymbol\alpha|\le K$. It drops a
  monomial above that degree.
- **`PSCarlemanBasis`**: the same monomials. Instead of dropping a monomial above the degree, it
  re-expands that monomial about a pivot $s$,
  $y_{\boldsymbol\alpha}\mapsto y_{\boldsymbol\alpha}-\prod_i\left(u_i-s_i\right)^{\alpha_i}$.
- **`GaussianHermiteBasis`**: products of Hermite functions,
  $\psi_{\boldsymbol\alpha}(u)=\prod_i\psi_{\alpha_i}(u_i)$. These are square-integrable, so an
  expectation value is a well-defined integral, and `Liouville` and `KoopmanVonNeumann`
  accept it.

One page each below. They follow the same seven sections, so reading one makes the others quick.

Both halves of the pipeline take the same basis object, so a matrix
([`05_matrix`](../05_matrix.ipynb)) and a vector ([`06_vector`](../06_vector.ipynb)) can never
disagree about the expansion.

## One basis state

Indexing a basis gives you one element of it. A sparse tuple of `(site, power)` factors names that
element. `(1, 2)` is the factor of site 1 raised to 2. Absent sites carry power 0, so the vacuum is
the empty tuple `()`.

The scheme is the same on every basis. Only what the numbers mean changes.

What comes back is a `CarlemanState` or a `GaussianHermiteState`, both subclasses of
`BasisFunction`. Either is a SymPy `Function`, so it prints readably and composes into expressions.
It is not an opaque handle.

In [1]:
import sympy as sp

from lin_ode.basis import CarlemanBasis, GaussianHermiteBasis, PSCarlemanBasis

pivot = sp.IndexedBase("s")  # per-site pivot; PSCarlemanBasis reads it as pivot[site]

# the same address on all three bases
print("CarlemanBasis        ", CarlemanBasis(3, 2)[(1, 2)])
print("PSCarlemanBasis      ", PSCarlemanBasis(3, 2, pivot)[(1, 2)])
print("GaussianHermiteBasis ", GaussianHermiteBasis(3, 2)[(1, 2)])

CarlemanBasis         CarlemanState((1, 2))
PSCarlemanBasis       CarlemanState((1, 2))
GaussianHermiteBasis  GaussianHermiteState((1, 2))


## The whole basis

A basis counts its states and enumerates them, and it converts between the structured index and a
**flat index** for building dense vectors and matrices.

| member | what it gives you |
| --- | --- |
| `len(basis)` | the number of states |
| `basis.n_dof` | the number of sites |
| `basis[structured]` | the basis state at that address |
| `basis.structured_idx_of_flat_index(k)` | flat $\to$ structured |
| `basis.flat_index_of_structured_index(idx)` | structured $\to$ flat |
| `basis.flat_index_of_basis_function(state)` | as above, from the object; `None` if out of basis |

`truncation_rule` and `truncate_expression` are the other two members, and you rarely call either.
`OperatorMatrix` and `Vector` apply them for you when a state goes above the degree. Each page below
works through its own in sections 5 and 6.

`is_l2` is a flag you read rather than a member you call. `Liouville` and `KoopmanVonNeumann`
transport a state, so they need a square-integrable basis, and `OperatorMatrix` raises
`IncompatibleBasisError` without one ([`03_linearization`](../03_linearization.ipynb)).

Every basis subclasses `TruncatedBasisSet`. You never construct that one, but it is the type to
write when a helper should take any basis.

In [2]:
basis = CarlemanBasis(n_vars=3, degree=2)

print("n_dof     ", basis.n_dof)
print("len       ", len(basis), " = C(n_dof + degree, degree)")
print("is_l2     ", basis.is_l2)
print()
for flat in range(len(basis)):
    print(f"  flat {flat}  <->  {str(basis.structured_idx_of_flat_index(flat)):24}"
          f"  {basis[basis.structured_idx_of_flat_index(flat)]}")

n_dof      3
len        10  = C(n_dof + degree, degree)
is_l2      False

  flat 0  <->  ()                        CarlemanState()
  flat 1  <->  ((0, 1),)                 CarlemanState((0, 1))
  flat 2  <->  ((1, 1),)                 CarlemanState((1, 1))
  flat 3  <->  ((2, 1),)                 CarlemanState((2, 1))
  flat 4  <->  ((0, 2),)                 CarlemanState((0, 2))
  flat 5  <->  ((0, 1), (1, 1))          CarlemanState((0, 1), (1, 1))
  flat 6  <->  ((0, 1), (2, 1))          CarlemanState((0, 1), (2, 1))
  flat 7  <->  ((1, 2),)                 CarlemanState((1, 2))
  flat 8  <->  ((1, 1), (2, 1))          CarlemanState((1, 1), (2, 1))
  flat 9  <->  ((2, 2),)                 CarlemanState((2, 2))


### The flat ordering

States are ordered by **total degree**, then lexicographically within each degree. For $N$ sites
truncated at degree $P$:

- $k=0$ is the vacuum.
- Then the $N$ degree-1 states, in ascending site order.
- Then the degree-2 states in `combinations_with_replacement` order: $u_0^2,\ u_0u_1,\ \dots$
- And so on to degree $P$.

Degree $D$ contributes $\binom{N+D-1}{D}$ states, so its block starts at

$$\text{offset}(D)=1+\sum_{k=1}^{D-1}\binom{N+k-1}{k},
\qquad \text{and} \qquad
\operatorname{len}(\text{basis})=\binom{N+P}{P}.$$

The two index functions are exact inverses, so
`flat_index_of_structured_index(structured_idx_of_flat_index(k)) == k` for every valid $k$.

In [3]:
from itertools import combinations_with_replacement
from math import comb

# the same basis enumerated above
for degree in range(basis.degree + 1):
    block = comb(basis.n_dof + degree - 1, degree) if degree else 1
    offset = 1 + sum(comb(basis.n_dof + k - 1, k) for k in range(1, degree)) if degree else 0
    states = [basis.structured_idx_of_flat_index(offset + j) for j in range(block)]
    print(f"degree {degree}:  k = {offset}..{offset + block - 1}   {states}")

# the order is combinations_with_replacement over site indices, graded by degree
expected = [()] + [
    tuple(sorted({s: combo.count(s) for s in combo}.items()))
    for degree in range(1, basis.degree + 1)
    for combo in combinations_with_replacement(range(basis.n_dof), degree)
]
assert expected == [basis.structured_idx_of_flat_index(k) for k in range(len(basis))]
assert all(basis.flat_index_of_structured_index(basis.structured_idx_of_flat_index(k)) == k
           for k in range(len(basis)))
print(f"\n{len(basis)} states = C({basis.n_dof} + {basis.degree}, {basis.degree}); "
      "ordering and round-trip verified")

degree 0:  k = 0..0   [()]
degree 1:  k = 1..3   [((0, 1),), ((1, 1),), ((2, 1),)]
degree 2:  k = 4..9   [((0, 2),), ((0, 1), (1, 1)), ((0, 1), (2, 1)), ((1, 2),), ((1, 1), (2, 1)), ((2, 2),)]

10 states = C(3 + 2, 2); ordering and round-trip verified


On a multi-dimensional grid the site label is a tuple. `lin_ode` flattens those row-major before
applying the ordering above, so the last axis varies fastest.

In [4]:
plate = CarlemanBasis(n_vars=(2, 3), degree=1)

for k in range(len(plate)):
    print(f"  k={k}  {plate.structured_idx_of_flat_index(k)}")

  k=0  ()
  k=1  (((0, 0), 1),)
  k=2  (((0, 1), 1),)
  k=3  (((0, 2), 1),)
  k=4  (((1, 0), 1),)
  k=5  (((1, 1), 1),)
  k=6  (((1, 2), 1),)


```{toctree}
:maxdepth: 1

carleman
psc_carleman
gaussian_hermite
```

---

[← 2. Linearization](../03_linearization.ipynb) · [Notebook index](../../README.md) · [CarlemanBasis →](carleman.ipynb)